# Assignment 1 — Deep Learning for Perceptron

Images needed in the same folder: `dog.jpg`, `shelf.jpg`, `template.jpg`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from numpy.lib.stride_tricks import as_strided

np.random.seed(42)

In [ ]:
def load_gray(path):
    return np.array(Image.open(path).convert('L'), dtype=np.float64)


def convolve2d(image, kernel):
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant')
    h, w = image.shape
    s0, s1 = padded.strides
    # build a view of all (kh x kw) patches without copying
    patches = as_strided(padded, shape=(h, w, kh, kw), strides=(s0, s1, s0, s1))
    # flip kernel 180° (convolution def) then dot with each patch
    return np.einsum('ijkl,kl->ij', patches, kernel[::-1, ::-1])


def correlate2d(image, kernel):
    # same as convolve2d but no flip
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant')
    h, w = image.shape
    s0, s1 = padded.strides
    patches = as_strided(padded, shape=(h, w, kh, kw), strides=(s0, s1, s0, s1))
    return np.einsum('ijkl,kl->ij', patches, kernel)


def gaussian_kernel(size, sigma):
    k = size // 2
    y, x = np.mgrid[-k:k+1, -k:k+1]
    g = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    return g / g.sum()


def show(images, titles, cmap='gray', figsize=None):
    n = len(images)
    figsize = figsize or (5 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Load all three images
dog      = load_gray('dog.jpg')
shelf    = load_gray('shelf.jpg')
template = load_gray('template.jpg')

print(f'dog      : {dog.shape}')
print(f'shelf    : {shelf.shape}')
print(f'template : {template.shape}')

## Task 1 — Adding Gaussian Noise

In [ ]:
noise = np.random.normal(loc=0, scale=15, size=dog.shape)
noisy_dog = np.clip(dog + noise, 0, 255)

print(f'noise mean: {noise.mean():.4f}, std: {noise.std():.4f}')
show([dog, noisy_dog], ['Original', 'Noisy (μ=0, σ=15)'])

## Task 2 — Convolution (Sobel-X kernel)

In [ ]:
sobel_kernel = np.array([[1,  0, -1],
                         [2,  0, -2],
                         [1,  0, -1]], dtype=np.float64)

sobel_out = convolve2d(dog, sobel_kernel)
sobel_display = np.clip(np.abs(sobel_out), 0, 255)

show([dog, sobel_display], ['Original', 'Sobel-X output (|conv|)'])
print(f'output range: [{sobel_out.min():.1f}, {sobel_out.max():.1f}]')

## Task 3 — Denoising with Gaussian Filter + Sharpening

In [ ]:
# 7x7 Gaussian kernel, sigma=1.0
gauss_k = gaussian_kernel(size=7, sigma=1.0)
print(gauss_k.round(5))
print(f'sum: {gauss_k.sum():.8f}')

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(gauss_k, cmap='viridis')
plt.colorbar(im, ax=ax)
ax.set_title('Gaussian kernel (7x7, σ=1.0)')
plt.tight_layout()
plt.show()

denoised = np.clip(convolve2d(noisy_dog, gauss_k), 0, 255)
show([noisy_dog, denoised], ['Noisy', 'Denoised'])

In [ ]:
sharpening_kernel = np.array([
    [1,  4,   6,  4, 1],
    [4, 16,  24, 16, 4],
    [6, 24, -476, 24, 6],
    [4, 16,  24, 16, 4],
    [1,  4,   6,  4, 1],
], dtype=np.float64) / (-256.0)

print(f'kernel sum: {sharpening_kernel.sum():.6f}')

sharpened = np.clip(convolve2d(denoised, sharpening_kernel), 0, 255)
show([dog, denoised, sharpened], ['Original', 'Denoised', 'Sharpened'], figsize=(15, 4))

## Task 4 — Template Matching: Convolution vs Correlation

Both methods slide a template over the shelf image and compute a similarity score at each position. Mean subtraction (from both image and template) stops bright regions from getting artificially high scores.

In [ ]:
def _sliding_scores(image, kernel):
    # valid-mode sliding dot product using stride tricks (no padding)
    H, W = image.shape
    kh, kw = kernel.shape
    out_h = H - kh + 1
    out_w = W - kw + 1
    scores = np.zeros((out_h, out_w))
    for i in range(out_h):
        row_block = image[i:i+kh, :]
        s0, s1 = row_block.strides
        col_patches = as_strided(row_block, shape=(out_w, kh, kw), strides=(s1, s0, s1))
        scores[i] = np.einsum('jkl,kl->j', col_patches, kernel)
    return scores


def match_conv(image, template):
    # pre-flip template before passing to convolution so the internal flip cancels out
    img  = image.astype(float) - image.mean()
    tmpl = template.astype(float) - template.mean()
    return _sliding_scores(img, tmpl[::-1, ::-1][::-1, ::-1])


def match_corr(image, template):
    # correlation: template used as-is, no flip
    img  = image.astype(float) - image.mean()
    tmpl = template.astype(float) - template.mean()
    return _sliding_scores(img, tmpl)

In [ ]:
conv_scores = match_conv(shelf, template)
corr_scores = match_corr(shelf, template)

conv_loc = np.unravel_index(np.argmax(conv_scores), conv_scores.shape)
corr_loc = np.unravel_index(np.argmax(corr_scores), corr_scores.shape)

print(f'conv peak: {conv_loc}')
print(f'corr peak: {corr_loc}')
print(f'same result: {conv_loc == corr_loc}')

In [ ]:
th, tw = template.shape
r, c   = conv_loc
r2, c2 = corr_loc

fig, axes = plt.subplots(2, 2, figsize=(20, 8))

axes[0, 0].imshow(shelf, cmap='gray')
axes[0, 0].set_title('Shelf')
axes[0, 0].axis('off')

axes[0, 1].imshow(template, cmap='gray')
axes[0, 1].set_title('Template')
axes[0, 1].axis('off')

axes[1, 0].imshow(conv_scores, cmap='hot')
axes[1, 0].add_patch(mpatches.Rectangle((c, r), tw, th, lw=2, edgecolor='cyan', facecolor='none'))
axes[1, 0].set_title(f'Convolution scores  (peak: {r}, {c})')
axes[1, 0].axis('off')

axes[1, 1].imshow(corr_scores, cmap='hot')
axes[1, 1].add_patch(mpatches.Rectangle((c2, r2), tw, th, lw=2, edgecolor='cyan', facecolor='none'))
axes[1, 1].set_title(f'Correlation scores  (peak: {r2}, {c2})')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# overlay on shelf
fig2, ax2 = plt.subplots(figsize=(18, 4))
ax2.imshow(shelf, cmap='gray')
ax2.add_patch(mpatches.Rectangle((c2, r2), tw, th, lw=3, edgecolor='red', facecolor='none'))
ax2.set_title(f'Match at row={r2}, col={c2}')
ax2.axis('off')
plt.tight_layout()
plt.show()

## Task 4 — Discussion

**e. Accuracy:** Both methods find the exact same peak position. Pre-flipping the template before convolution cancels the internal flip, so the score at every pixel is identical to correlation. No accuracy difference.

**f. Efficiency:** Same asymptotic cost. Correlation is slightly simpler because there's no pre-flip step. For large images, FFT-based matching would be the practical choice over either direct approach.

**g. Which is better suited?** Correlation is the more natural fit. Template matching is essentially "how similar is this patch to the template" — a direct dot product — which is exactly what correlation computes. Convolution is built around the idea of flipping the kernel, which makes sense for filters (blur, edge detection) but is just extra overhead for matching a known template. You pre-flip to undo the flip, which works, but it's unnecessary complexity. Correlation is cleaner and more intuitive for this use case.